# Bronze → Silver ETL
## Enterprise Access Governance — Medallion Architecture

**Purpose:** Read raw CSV files from the Bronze layer (`Files/bronze/`), apply data quality checks, enforce schemas, add audit columns, and write cleaned Delta Parquet tables to the Silver layer (`Tables/silver_*`).

**Author:** Enterprise Access Governance Project  
**Layer:** Silver (Cleaned & Validated)

---
### Tables Produced
| Silver Table | Source Bronze File | Description |
|---|---|---|
| `silver_departments` | departments.csv | Cleaned department hierarchy |
| `silver_employees` | employees.csv | Validated employee roster |
| `silver_security_groups` | security_groups.csv | Security group definitions |
| `silver_resources` | resources.csv | Resource inventory |
| `silver_group_memberships` | group_memberships.csv | Employee→Group edges |
| `silver_group_permissions` | group_permissions.csv | Group→Resource edges |
| `silver_group_inheritance` | group_inheritance.csv | Group→Group edges |

---
## Cell 1 — Setup & Imports

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DateType, TimestampType
)
from datetime import datetime

# ── Configuration ──────────────────────────────────────────────
BRONZE_PATH  = "Files/bronze"          # Raw files uploaded via Lakehouse UI
SILVER_DB    = "silver"                # Will prefix all silver table names
INGESTED_AT  = datetime.utcnow().isoformat() + "Z"   # Pipeline run timestamp
SOURCE_TAG   = "bronze_csv_upload_v1"

print(f"Pipeline started at: {INGESTED_AT}")
print(f"Bronze path : {BRONZE_PATH}")
print(f"Silver DB   : {SILVER_DB}")

---
## Cell 2 — Helper Functions

In [ ]:
def read_bronze_csv(filename: str) -> DataFrame:
    """Read a CSV from the bronze Files folder with header + infer schema."""
    path = f"{BRONZE_PATH}/{filename}"
    df = spark.read.option("header", True).option("inferSchema", True).csv(path)
    print(f"Read {filename}: {df.count()} rows, {len(df.columns)} columns")
    return df


def add_audit_columns(df: DataFrame, source_file: str) -> DataFrame:
    """Append standard audit/lineage columns to every silver table."""
    return (
        df
        .withColumn("_ingested_at",   F.lit(INGESTED_AT).cast(TimestampType()))
        .withColumn("_source_file",   F.lit(source_file))
        .withColumn("_pipeline_tag",  F.lit(SOURCE_TAG))
    )


def write_silver_table(df: DataFrame, table_name: str) -> None:
    """Write a DataFrame as a managed Delta table in the Silver layer."""
    full_name = f"silver_{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    count = spark.table(full_name).count()
    print(f"Written silver_{table_name}: {count} rows")


def assert_no_nulls(df: DataFrame, column: str, table_name: str) -> None:
    """Data quality gate: fail loudly if a primary key column has nulls."""
    null_count = df.filter(F.col(column).isNull()).count()
    if null_count > 0:
        raise ValueError(
            f"DQ FAIL [{table_name}]: Column '{column}' has {null_count} NULL values!"
        )
    print(f"   DQ PASS [{table_name}]: '{column}' has 0 NULLs")


def assert_no_duplicates(df: DataFrame, column: str, table_name: str) -> None:
    """Data quality gate: fail loudly if a primary key column has duplicates."""
    total   = df.count()
    distinct = df.select(column).distinct().count()
    if total != distinct:
        raise ValueError(
            f"DQ FAIL [{table_name}]: Column '{column}' has {total - distinct} duplicate values!"
        )
    print(f"   DQ PASS [{table_name}]: '{column}' has 0 duplicates ({distinct} unique)")


def assert_valid_values(df: DataFrame, column: str, valid_values: list, table_name: str) -> None:
    """Data quality gate: fail if any row has an unexpected categorical value."""
    invalid_count = df.filter(~F.col(column).isin(valid_values)).count()
    if invalid_count > 0:
        raise ValueError(
            f"DQ FAIL [{table_name}]: Column '{column}' has {invalid_count} invalid values! "
            f"Expected one of: {valid_values}"
        )
    print(f"   DQ PASS [{table_name}]: '{column}' has all valid values")


print("Helper functions defined")

---
## Cell 3 — Silver: Departments

In [ ]:
print("=" * 50)
print("Processing: DEPARTMENTS")
print("=" * 50)

df_dept_raw = read_bronze_csv("departments.csv")
df_dept_raw.printSchema()

# ── Clean ──────────────────────────────────────────────────────
df_dept = (
    df_dept_raw
    .withColumn("department_id",        F.trim(F.col("department_id")))
    .withColumn("name",                 F.trim(F.col("name")))
    .withColumn("cost_center",          F.trim(F.col("cost_center")))
    .withColumn("parent_department_id", F.trim(F.col("parent_department_id")))
    # Standardise nulls — root department has no parent
    .withColumn("parent_department_id",
        F.when(F.col("parent_department_id") == "None", None)
         .otherwise(F.col("parent_department_id"))
    )
    .dropDuplicates(["department_id"])
)

# ── Data Quality ───────────────────────────────────────────────
assert_no_nulls(df_dept, "department_id", "departments")
assert_no_duplicates(df_dept, "department_id", "departments")
assert_no_nulls(df_dept, "name", "departments")

# ── Add Audit Columns & Write ──────────────────────────────────
df_dept = add_audit_columns(df_dept, "departments.csv")
write_silver_table(df_dept, "departments")
df_dept.show(5, truncate=False)

---
## Cell 4 — Silver: Employees

In [ ]:
print("=" * 50)
print("Processing: EMPLOYEES")
print("=" * 50)

df_emp_raw = read_bronze_csv("employees.csv")
df_emp_raw.printSchema()

# ── Clean ──────────────────────────────────────────────────────
df_emp = (
    df_emp_raw
    .withColumn("employee_id",    F.trim(F.col("employee_id")))
    .withColumn("name",           F.trim(F.col("name")))
    .withColumn("email",          F.lower(F.trim(F.col("email"))))
    .withColumn("department_id",  F.trim(F.col("department_id")))
    .withColumn("employee_type",  F.trim(F.col("employee_type")))
    .withColumn("hire_date",      F.to_date(F.col("hire_date"), "yyyy-MM-dd"))
    .withColumn("manager_id",
        F.when(F.col("manager_id") == "None", None)
         .otherwise(F.trim(F.col("manager_id")))
    )
    .dropDuplicates(["employee_id"])
    .filter(F.col("employee_id").isNotNull())
)

# ── Data Quality ───────────────────────────────────────────────
assert_no_nulls(df_emp, "employee_id", "employees")
assert_no_duplicates(df_emp, "employee_id", "employees")
assert_no_nulls(df_emp, "name", "employees")
assert_no_nulls(df_emp, "email", "employees")
assert_valid_values(
    df_emp, "employee_type",
    ["Full-Time", "Contractor", "Vendor"],
    "employees"
)

# ── Print Distribution ─────────────────────────────────────────
print("\nEmployee Type Distribution:")
df_emp.groupBy("employee_type").count().orderBy("count", ascending=False).show()

# ── Add Audit Columns & Write ──────────────────────────────────
df_emp = add_audit_columns(df_emp, "employees.csv")
write_silver_table(df_emp, "employees")
df_emp.show(5, truncate=False)

---
## Cell 5 — Silver: Security Groups

In [ ]:
print("=" * 50)
print("Processing: SECURITY GROUPS")
print("=" * 50)

df_grp_raw = read_bronze_csv("security_groups.csv")
df_grp_raw.printSchema()

# ── Clean ──────────────────────────────────────────────────────
df_grp = (
    df_grp_raw
    .withColumn("group_id",        F.trim(F.col("group_id")))
    .withColumn("group_name",      F.trim(F.col("group_name")))
    .withColumn("description",     F.trim(F.col("description")))
    .withColumn("parent_group_id",
        F.when(F.col("parent_group_id") == "None", None)
         .otherwise(F.trim(F.col("parent_group_id")))
    )
    .dropDuplicates(["group_id"])
    .filter(F.col("group_id").isNotNull())
)

# ── Data Quality ───────────────────────────────────────────────
assert_no_nulls(df_grp, "group_id", "security_groups")
assert_no_duplicates(df_grp, "group_id", "security_groups")
assert_no_nulls(df_grp, "group_name", "security_groups")

# ── Add Audit Columns & Write ──────────────────────────────────
df_grp = add_audit_columns(df_grp, "security_groups.csv")
write_silver_table(df_grp, "security_groups")
df_grp.show(5, truncate=False)

---
## Cell 6 — Silver: Resources

In [ ]:
print("=" * 50)
print("Processing: RESOURCES")
print("=" * 50)

df_res_raw = read_bronze_csv("resources.csv")
df_res_raw.printSchema()

# ── Clean ──────────────────────────────────────────────────────
df_res = (
    df_res_raw
    .withColumn("resource_id",          F.trim(F.col("resource_id")))
    .withColumn("resource_name",        F.trim(F.col("resource_name")))
    .withColumn("resource_type",        F.trim(F.col("resource_type")))
    .withColumn("sensitivity",          F.trim(F.col("sensitivity")))
    .withColumn("owner_department_id",  F.trim(F.col("owner_department_id")))
    .dropDuplicates(["resource_id"])
    .filter(F.col("resource_id").isNotNull())
)

# ── Data Quality ───────────────────────────────────────────────
assert_no_nulls(df_res, "resource_id", "resources")
assert_no_duplicates(df_res, "resource_id", "resources")
assert_valid_values(
    df_res, "resource_type",
    ["Database", "Server", "Application", "FileShare"],
    "resources"
)
assert_valid_values(
    df_res, "sensitivity",
    ["Low", "Medium", "High", "Critical"],
    "resources"
)

# ── Print Sensitivity Distribution ────────────────────────────
print("\nResource Sensitivity Distribution:")
df_res.groupBy("sensitivity").count().orderBy("count", ascending=False).show()

# ── Add Audit Columns & Write ──────────────────────────────────
df_res = add_audit_columns(df_res, "resources.csv")
write_silver_table(df_res, "resources")
df_res.show(5, truncate=False)

---
## Cell 7 — Silver: Group Memberships

In [ ]:
print("=" * 50)
print("Processing: GROUP MEMBERSHIPS")
print("=" * 50)

df_mem_raw = read_bronze_csv("group_memberships.csv")
df_mem_raw.printSchema()

# ── Clean ──────────────────────────────────────────────────────
df_mem = (
    df_mem_raw
    .withColumn("employee_id",    F.trim(F.col("employee_id")))
    .withColumn("group_id",       F.trim(F.col("group_id")))
    .withColumn("assigned_date",  F.to_date(F.col("assigned_date"), "yyyy-MM-dd"))
    .withColumn("assigned_by",    F.trim(F.col("assigned_by")))
    # Composite key dedup
    .dropDuplicates(["employee_id", "group_id"])
    .filter(F.col("employee_id").isNotNull() & F.col("group_id").isNotNull())
)

# ── Data Quality ───────────────────────────────────────────────
assert_no_nulls(df_mem, "employee_id", "group_memberships")
assert_no_nulls(df_mem, "group_id",    "group_memberships")

# Referential integrity: every employee_id must exist in silver_employees
df_emp_ids  = spark.table("silver_employees").select("employee_id")
df_grp_ids  = spark.table("silver_security_groups").select("group_id")

orphan_employees = df_mem.join(df_emp_ids, "employee_id", "left_anti").count()
orphan_groups    = df_mem.join(df_grp_ids, "group_id",    "left_anti").count()

if orphan_employees > 0:
    print(f"WARNING: {orphan_employees} memberships reference unknown employee_ids")
else:
    print("   DQ PASS [group_memberships]: All employee_ids are valid")

if orphan_groups > 0:
    print(f"WARNING: {orphan_groups} memberships reference unknown group_ids")
else:
    print("   DQ PASS [group_memberships]: All group_ids are valid")

# ── Add Audit Columns & Write ──────────────────────────────────
df_mem = add_audit_columns(df_mem, "group_memberships.csv")
write_silver_table(df_mem, "group_memberships")
df_mem.show(5, truncate=False)

---
## Cell 8 — Silver: Group Permissions

In [ ]:
print("=" * 50)
print("Processing: GROUP PERMISSIONS")
print("=" * 50)

df_perm_raw = read_bronze_csv("group_permissions.csv")
df_perm_raw.printSchema()

# ── Clean ──────────────────────────────────────────────────────
df_perm = (
    df_perm_raw
    .withColumn("group_id",        F.trim(F.col("group_id")))
    .withColumn("resource_id",     F.trim(F.col("resource_id")))
    .withColumn("permission_level",F.trim(F.col("permission_level")))
    .withColumn("granted_date",    F.to_date(F.col("granted_date"), "yyyy-MM-dd"))
    .dropDuplicates(["group_id", "resource_id"])
    .filter(F.col("group_id").isNotNull() & F.col("resource_id").isNotNull())
)

# ── Data Quality ───────────────────────────────────────────────
assert_no_nulls(df_perm, "group_id",    "group_permissions")
assert_no_nulls(df_perm, "resource_id", "group_permissions")
assert_valid_values(
    df_perm, "permission_level",
    ["Read", "Write", "Admin"],
    "group_permissions"
)

# ── Print Permission Distribution ──────────────────────────────
print("\nPermission Level Distribution:")
df_perm.groupBy("permission_level").count().orderBy("count", ascending=False).show()

# ── Add Audit Columns & Write ──────────────────────────────────
df_perm = add_audit_columns(df_perm, "group_permissions.csv")
write_silver_table(df_perm, "group_permissions")
df_perm.show(5, truncate=False)

---
## Cell 9 — Silver: Group Inheritance

In [ ]:
print("=" * 50)
print("Processing: GROUP INHERITANCE")
print("=" * 50)

df_inh_raw = read_bronze_csv("group_inheritance.csv")
df_inh_raw.printSchema()

# ── Clean ──────────────────────────────────────────────────────
df_inh = (
    df_inh_raw
    .withColumn("child_group_id",  F.trim(F.col("child_group_id")))
    .withColumn("parent_group_id", F.trim(F.col("parent_group_id")))
    # Remove self-loops (data quality)
    .filter(F.col("child_group_id") != F.col("parent_group_id"))
    .dropDuplicates(["child_group_id", "parent_group_id"])
    .filter(F.col("child_group_id").isNotNull() & F.col("parent_group_id").isNotNull())
)

# ── Data Quality ───────────────────────────────────────────────
assert_no_nulls(df_inh, "child_group_id",  "group_inheritance")
assert_no_nulls(df_inh, "parent_group_id", "group_inheritance")

# ── Add Audit Columns & Write ──────────────────────────────────
df_inh = add_audit_columns(df_inh, "group_inheritance.csv")
write_silver_table(df_inh, "group_inheritance")
df_inh.show(truncate=False)

---
## Cell 10 — Final Silver Layer Validation Report

In [ ]:
print("=" * 60)
print("SILVER LAYER — FINAL VALIDATION REPORT")
print("=" * 60)

silver_tables = [
    "silver_departments",
    "silver_employees",
    "silver_security_groups",
    "silver_resources",
    "silver_group_memberships",
    "silver_group_permissions",
    "silver_group_inheritance",
]

for tbl in silver_tables:
    count = spark.table(tbl).count()
    print(f"  {tbl:<35} → {count:>5} rows")

# ── Contractor count for validation ───────────────────────────
contractor_count = spark.table("silver_employees") \
    .filter(F.col("employee_type") == "Contractor") \
    .count()

high_crit_count = spark.table("silver_resources") \
    .filter(F.col("sensitivity").isin(["High", "Critical"])) \
    .count()

print()
print(f"  Contractors in dataset     : {contractor_count}")
print(f"  High/Critical resources    : {high_crit_count}")
print()
print("Silver layer complete. Ready to build Gold layer.")